In [0]:
"""

===============================================================================
Procedure: Load Silver Table for customer az information (Bronze -> Silver)
===============================================================================
Script Purpose:
    This stored procedure performs the ETL (Extract, Transform, Load) process to 
    populate the 'silver' schema tables from the 'bronze' schema.
	Actions Performed:to full refresh the ETL patterns
		- Truncates Silver tables if already there.
		- Inserts transformed and cleansed data from Bronze into Silver tables.
	Objectives:
		1-Replace the "-" with "" for the cleanliness 
		2-Change abbrivation to the descriptive name 

"""

In [0]:
#init 

catalog_name = "abhi_dwh_sql_based"
source_schema  = "bronze"
sink_schema = "silver"
table_name = "erp_loc_a101"


# Read the erp cust az12 from the bronze layer 

In [0]:
df = spark.read.table(f"{catalog_name}.{source_schema}.{table_name}")



In [0]:
df.show(5)

## Transformation to clean the data

In [0]:
#import necessory library 

from pyspark.sql.functions import col, row_number, replace, regexp_replace, substring, length, coalesce, lit, upper, when, trim

### 0. Trim all the leading and trailing space from all the columns 

In [0]:
df = df.select([trim(col(c)).alias(c) if dict(df.dtypes)[c]=='string' else col(c) for c in df.columns])

### 1.Remve the abbrivation and add the descriptive names

In [0]:
df.select(col("CNTRY")).distinct().show()

In [0]:
cntry_map = {"DE":"Germany", "US":"United Stated", "USA":"United Stated", "":"n/a"}

expr=None
for k,v in cntry_map.items():
    expr = when(upper(col("CNTRY"))==k,v) if expr is None else expr.when(upper(col("CNTRY"))==k,v)

df = df.withColumn("CNTRY", expr.otherwise(col("CNTRY")))
                   
df = df.withColumn("CNTRY",coalesce(col("CNTRY"),lit("n/a")))
df.show()

In [0]:
df.select(col("CNTRY")).distinct().show()

# Write it to Silver Layer after cleansing 

In [0]:
df.write.mode("overwrite").saveAsTable(f"{catalog_name}.{sink_schema}.{table_name}")
print(f"{catalog_name}.{sink_schema}.{table_name} written successfully")
df.show()